Match all the data together
Produce data files for subsequent use

In [3]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import myfunction as mf

path_country_nc = "C:/Users/dell/OneDrive/file/nc/"
path_one_spdb = 'C:/Users/dell/OneDrive/file/SPDB/'
path_inf = 'C:/Users/dell/OneDrive/file/'

meta_file = 'meta_data.csv'
inf_file = 'inf.csv'

drive_letter = 'E:'
path_part = "part_data"
path_part0 = "part0_treat"
path_med ="/wyy/code_project/running_outcome/final_data/SPDB/"
path_set = drive_letter + path_med + path_part0 + "/"

path_data_raw = drive_letter + path_med + path_part + "/"

path_pre = path_set + "pretreatment/"
path_match = path_set + "match/"
path_picture = path_set + "fig/"
path_temp = path_set + "temp/"

list_po_15 = ['PFNA', 'PFHpA', 'PFOA', 'PFOS', 'FOSA', 'PFDA', 'PFBS', 'PFBA', 
              'PFHxA', 'PFHxS', 'PFTeDA', 'PFPeA', 'PFDoDA', 'PFTrDA', 'PFUnDA']


str_sw_remark = ''

list_color = ["#ee877c", "#8bd0e3", "#6abeae", "#808eaf", "#f7bba8", "#acb4cc", "#b5e0d5", "#e86462", "#a89687"]

In [ ]:


df_data_lr = pd.read_csv(path_match + "lr_data_po.csv")
df_data_sw = pd.read_csv(path_match + "sw_data_po.csv")

dict_inf = {"pon":["po_classification"]}

df_lr_add = mf.append_inf(df_data_lr, dict_inf)

df_sw_add = mf.append_inf(df_data_sw, dict_inf)
pd.set_option('display.max_rows', None)
list_items = ['posname', 'po_classification']
for item in list_items:

    df_item = mf.col_value(df_lr_add,item)
    print(df_item)

for item in list_items:

    df_item = mf.col_value(df_sw_add,item)
    print(df_item)


          posname  count       per
0            PFOS   1992  0.092278
1            PFDA   1372  0.063557
2          PFDoDA   1343  0.062213
3          PFUnDA   1340  0.062074
4            PFNA   1309  0.060638
5            PFOA   1289  0.059712
6           PFHxS   1139  0.052763
7           PFHpA   1086  0.050308
8           PFHxA   1066  0.049382
9            PFBS   1029  0.047668
10           FOSA    996  0.046139
11          PFPeA    996  0.046139
12           PFBA    895  0.041460
13         PFTrDA    713  0.033029
14         PFTeDA    656  0.030389
15           PFDS    538  0.024922
16          PFHpS    364  0.016862
17       8:2 FTSA    352  0.016306
18        MeFOSAA    350  0.016213
19          PFPeS    346  0.016028
20       6:2 FTSA    345  0.015982
21        EtFOSAA    344  0.015936
22           PFNS    336  0.015565
23       4:2 FTSA    321  0.014870
24         MeFOSA    284  0.013156
25        HFPO-DA    278  0.012878
26         EtFOSA    272  0.012600
27         PFHxDA   

In [ ]:
df_lr_treat = pd.read_csv(path_pre + "lr_treat.csv")

df_sw_treat = pd.read_csv(path_pre + "sw_treat.csv")

dict_inf_lr = {"pot":["po_carbon","po_f_carbon","po_chain","po_m_w","logPx","log_Koc", "log_Kow","log_Kaw",
                    "log_Koa_wet","log_KHxd_air","log_Koil_w","log_Koil_air","density","melting_point",
                    "boiling_point","solubility","log_pKa", "logD5_5", "logD7_4"]
                    ,"spt":["sp_length","sp_weight","sp_troph"]}
dict_inf_sw = {"pot":["po_carbon","po_f_carbon","po_chain","po_m_w","logPx","log_Koc", "log_Kow","log_Kaw",
                    "log_Koa_wet","log_KHxd_air","log_Koil_w","log_Koil_air","density","melting_point",
                    "boiling_point","solubility","log_pKa", "logD5_5", "logD7_4"]}

df_lr_add = mf.append_inf(df_lr_treat, dict_inf_lr)
df_sw_add = mf.append_inf(df_sw_treat, dict_inf_sw)



df_lr_add = df_lr_add[df_lr_add['sp_length'].notna()]

columns_to_drop = ['habitat']
df_lr_add = df_lr_add.drop(columns=columns_to_drop)

df_geo = pd.read_csv(path_pre + "geo_global_treat.csv")

df_lr_all = pd.merge(df_geo, df_lr_add, on=['lat_grid', 'lon_grid', 'year'],how='right')
df_sw_all = pd.merge(df_geo, df_sw_add, on=['lat_grid', 'lon_grid', 'year'],how='right')

print(df_lr_all.isnull().sum()[df_lr_all.isnull().sum() > 0])
print(df_sw_all.isnull().sum()[df_sw_all.isnull().sum() > 0])


df_sw_all_copy = df_sw_all.copy()


df_sw_all_copy.rename(columns={'value': 'sw_value'}, inplace=True)
df_sw_all_copy = df_sw_all_copy[['lat_grid', 'lon_grid', 'posname', 'year', 'sw_value']]
df_lr_sw = pd.merge(df_lr_all, df_sw_all_copy, on=['lat_grid', 'lon_grid', 'posname', 'year'], how='left')
df_lr_sw = df_lr_sw[df_lr_sw['sw_value'].notna()]


df_lr_analysis = df_lr_all.rename(columns={'type': 'habitat'})
df_sw_analysis = df_sw_all.rename(columns={'type': 'habitat'})
df_lr_sw_analysis = df_lr_sw.rename(columns={'type': 'habitat'})


df_lr_analysis = mf.select_data(df_lr_analysis, path_inf, 'bio')
df_sw_analysis = mf.select_data(df_sw_analysis, path_inf, 'w')
df_lr_sw_analysis = mf.select_data(df_lr_sw_analysis, path_inf, 'bio')

df_lr_analysis.to_csv(path_match + "lr_match_treat.csv", index=False)
df_sw_analysis.to_csv(path_match + "sw_match_treat.csv", index=False)
df_lr_sw_analysis.to_csv(path_match + "lr_sw_match_treat.csv", index=False)

Series([], dtype: int64)
Series([], dtype: int64)


In [ ]:
df_lr_treat = pd.read_csv(path_pre + "lr.csv")

df_sw_treat = pd.read_csv(path_pre + "sw.csv")

dict_inf_lr = {"pon":["po_carbon","po_f_carbon","po_chain","po_m_w","logPx","log_Koc", "log_Kow","log_Kaw",
                    "log_Koa_wet","log_KHxd_air","log_Koil_w","log_Koil_air","density","melting_point",
                    "boiling_point","solubility","log_pKa", "logD5_5", "logD7_4"]
                    ,"sp":["length_last","weight_last","troph_last"]}
dict_inf_sw = {"pon":["po_carbon","po_f_carbon","po_chain","po_m_w","logPx","log_Koc", "log_Kow","log_Kaw",
                    "log_Koa_wet","log_KHxd_air","log_Koil_w","log_Koil_air","density","melting_point",
                    "boiling_point","solubility","log_pKa", "logD5_5", "logD7_4"]}

df_lr_add = mf.append_inf(df_lr_treat, dict_inf_lr)
df_sw_add = mf.append_inf(df_sw_treat, dict_inf_sw)



df_lr_add.rename(columns={"length_last":"sp_length","weight_last":"sp_weight","troph_last":"sp_troph"},inplace=True)
df_lr_add = df_lr_add[df_lr_add['sp_length'].notna()]

columns_to_drop = ['habitat']
df_lr_add = df_lr_add.drop(columns=columns_to_drop)

df_geo = pd.read_csv(path_pre + "geo_global.csv")

df_lr_all = pd.merge(df_geo, df_lr_add, on=['lat_grid', 'lon_grid', 'year'],how='right')
df_sw_all = pd.merge(df_geo, df_sw_add, on=['lat_grid', 'lon_grid', 'year'],how='right')

print(df_lr_all.isnull().sum()[df_lr_all.isnull().sum() > 0])
print(df_sw_all.isnull().sum()[df_sw_all.isnull().sum() > 0])


df_sw_all_copy = df_sw_all.copy()


df_sw_all_copy.rename(columns={'value': 'sw_value'}, inplace=True)
df_sw_all_copy = df_sw_all_copy[['lat_grid', 'lon_grid', 'posname', 'year', 'sw_value']]
df_lr_sw = pd.merge(df_lr_all, df_sw_all_copy, on=['lat_grid', 'lon_grid', 'posname', 'year'], how='left')
df_lr_sw = df_lr_sw[df_lr_sw['sw_value'].notna()]

df_lr_analysis = df_lr_all.rename(columns={'type': 'habitat'})
df_sw_analysis = df_sw_all.rename(columns={'type': 'habitat'})
df_lr_sw_analysis = df_lr_sw.rename(columns={'type': 'habitat'})

df_lr_analysis = mf.select_data(df_lr_analysis, path_inf, 'bio')
df_sw_analysis = mf.select_data(df_sw_analysis, path_inf, 'w')
df_lr_sw_analysis = mf.select_data(df_lr_sw_analysis, path_inf, 'bio')


df_lr_analysis.to_csv(path_match + "lr_match.csv", index=False)
df_sw_analysis.to_csv(path_match + "sw_match.csv", index=False)
df_lr_sw_analysis.to_csv(path_match + "lr_sw_match.csv", index=False)

Series([], dtype: int64)
Series([], dtype: int64)
